# Task #18: inference + clause register

Runs the final legal-bert model from task #13 on the 102 test contracts and builds the clause
register. Pick a **T4 GPU runtime** first (Runtime > Change runtime type).

You need the task #13 weights: the `final_model/checkpoint/` folder from the Kaggle notebook
Output. Upload it to Google Drive and put its path in `CHECKPOINT_DIR` below. The folder should
have `encoder/`, `head.pt`, `tokenizer.json` and `classifier.json`.

See `docs/inference.md` for what the register is and how it was checked.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/Break-Through-Tech/Accenture-1T-contract-review-challenge.git"
REPO_REF = "task-18-inference"
REPO_DIR = Path("/content/contract-review")
CHECKPOINT_DIR = Path("/content/drive/MyDrive/contract-review/final/checkpoint")  # task #13 weights
OUT_DIR = Path("/content/drive/MyDrive/contract-review/inference")  # full register goes here

## 1. Drive, checkout, environment

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
OUT_DIR.mkdir(parents=True, exist_ok=True)
missing = [n for n in ("encoder", "head.pt", "classifier.json") if not (CHECKPOINT_DIR / n).exists()]
if missing:
    raise FileNotFoundError(f"{CHECKPOINT_DIR} is missing {missing}. Upload the task #13 checkpoint first.")

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--branch", REPO_REF, "--single-branch", REPO_URL, str(REPO_DIR)],
        check=True,
    )
subprocess.run([sys.executable, "-m", "pip", "install", "uv"], check=True)
UV = [sys.executable, "-m", "uv"]
subprocess.run(UV + ["sync", "--locked", "--dev", "--python", "3.12"], cwd=REPO_DIR, check=True)

def run_project(*args):
    # relay output line by line so Colab shows it
    process = subprocess.Popen(
        UV + ["run", "--locked", "--no-sync", *map(str, args)], cwd=REPO_DIR,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait():
        raise subprocess.CalledProcessError(process.returncode, args)

def script(*args):
    run_project("python", "-B", "-m", "scripts.inference", *args)

## 2. GPU check and tests

In [ ]:
run_project("python", "-c", "import torch; assert torch.cuda.is_available(), 'No GPU'; print(torch.cuda.get_device_name(0))")
run_project("python", "-B", "-m", "pytest", "-q", "-p", "no:cacheprovider", "tests/test_inference.py")

## 3. Run the model on the test contracts (a few minutes)

Chunks the 102 test contracts with legal-bert's tokenizer and scores every chunk. It also compares
the result with the probabilities task #13 saved. Expect `same_chunks_as_task13: true` and a
`max_difference_from_task13` close to 0 (small GPU differences are normal).

In [ ]:
PREDICTIONS = OUT_DIR / "test_chunk_predictions.parquet"
script("predict", "--checkpoint", CHECKPOINT_DIR, "--output", PREDICTIONS)

## 4. Build the clause register

Full register goes to Drive (`clause_register.csv` and `.parquet`). The sample and the check
against the gold clauses go to `docs/inference-results/` in the checkout; download them below.

In [ ]:
script("register", "--predictions", PREDICTIONS, "--output-dir", OUT_DIR)

In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive("/content/inference-results", "zip", root_dir=REPO_DIR / "docs" / "inference-results")
files.download(archive)